# Notebook 04 - Model Development
## Fake News Detection - NLP Assignment
### Person 2: W.A. Irusha Madushan (CIT-24-01-0514)
### Models: Random Forest (ML) + CNN (Deep Learning)

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# TensorFlow - CNN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully")
print("TensorFlow version:", tf.__version__)

All libraries imported successfully
TensorFlow version: 2.21.0


## 1. Loading Saved Features

The TF-IDF features and CNN padded sequences that were prepared 
in Notebook 03 are loaded here. This avoids repeating the 
feature engineering process and ensures both models are trained 
on exactly the same train/test split.

In [2]:
# Load TF-IDF features
X_train_tfidf = sp.load_npz('../models/X_train_tfidf.npz')
X_test_tfidf = sp.load_npz('../models/X_test_tfidf.npz')

# Load CNN padded sequences
X_train_pad = np.load('../models/X_train_pad.npy')
X_test_pad = np.load('../models/X_test_pad.npy')

# Load labels
y_train = np.load('../models/y_train.npy')
y_test = np.load('../models/y_test.npy')

print("All features loaded successfully")
print("TF-IDF Train shape:", X_train_tfidf.shape)
print("TF-IDF Test shape:", X_test_tfidf.shape)
print("CNN Train shape:", X_train_pad.shape)
print("CNN Test shape:", X_test_pad.shape)
print("Train labels shape:", y_train.shape)
print("Test labels shape:", y_test.shape)

All features loaded successfully
TF-IDF Train shape: (57632, 50000)
TF-IDF Test shape: (14409, 50000)
CNN Train shape: (57632, 300)
CNN Test shape: (14409, 300)
Train labels shape: (57632,)
Test labels shape: (14409,)


## 2. Random Forest Model

Random Forest is an ensemble machine learning method that builds 
multiple decision trees during training and combines their results 
to make a final prediction. It works well with high dimensional 
data like TF-IDF features because it can handle a large number 
of input features and select the most important ones automatically.

I will first train a baseline model, then experiment with 
different hyperparameters to find the best settings.

In [3]:
# Baseline Random Forest model
print("Training baseline Random Forest model...")
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train_tfidf, y_train)

# accuracy check
baseline_acc = accuracy_score(y_test, rf_baseline.predict(X_test_tfidf))
print("Baseline Random Forest Training Complete")
print("Baseline Accuracy:", round(baseline_acc * 100, 2), "%")

Training baseline Random Forest model...
Baseline Random Forest Training Complete
Baseline Accuracy: 95.21 %


## 3. Random Forest Hyperparameter Tuning

To improve the baseline model, I experimented with different 
numbers of trees (n_estimators) and maximum tree depth (max_depth). 
Comparing these settings helps find the best combination for 
fake news detection.

In [4]:
# checking with different hyperparameters
results = []

settings = [
    {'n_estimators': 100, 'max_depth': None},
    {'n_estimators': 200, 'max_depth': None},
    {'n_estimators': 100, 'max_depth': 50},
    {'n_estimators': 200, 'max_depth': 50},
]

for setting in settings:
    print(f"Training RF with n_estimators={setting['n_estimators']}, max_depth={setting['max_depth']}...")
    rf = RandomForestClassifier(
        n_estimators=setting['n_estimators'],
        max_depth=setting['max_depth'],
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test_tfidf))
    results.append({
        'n_estimators': setting['n_estimators'],
        'max_depth': setting['max_depth'],
        'accuracy': round(acc * 100, 2)
    })
    print(f"Accuracy: {round(acc * 100, 2)}%")

# Show results
results_df = pd.DataFrame(results)
print("\nHyperparameter Tuning Results:")
print(results_df)

Training RF with n_estimators=100, max_depth=None...
Accuracy: 95.21%
Training RF with n_estimators=200, max_depth=None...
Accuracy: 95.53%
Training RF with n_estimators=100, max_depth=50...
Accuracy: 94.82%
Training RF with n_estimators=200, max_depth=50...
Accuracy: 95.0%

Hyperparameter Tuning Results:
   n_estimators  max_depth  accuracy
0           100        NaN     95.21
1           200        NaN     95.53
2           100       50.0     94.82
3           200       50.0     95.00
